In [1]:
import pandas as pd

df = pd.read_csv('adult.data')
df.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
              'marital-status', 'occupation', 'relationship', 'race', 'sex', 
              'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
print(df.head())
print(df.shape)

   age          workclass  fnlwgt   education  education-num  \
0   50   Self-emp-not-inc   83311   Bachelors             13   
1   38            Private  215646     HS-grad              9   
2   53            Private  234721        11th              7   
3   28            Private  338409   Bachelors             13   
4   37            Private  284582     Masters             14   

        marital-status          occupation    relationship    race      sex  \
0   Married-civ-spouse     Exec-managerial         Husband   White     Male   
1             Divorced   Handlers-cleaners   Not-in-family   White     Male   
2   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
3   Married-civ-spouse      Prof-specialty            Wife   Black   Female   
4   Married-civ-spouse     Exec-managerial            Wife   White   Female   

   capital-gain  capital-loss  hours-per-week  native-country  income  
0             0             0              13   United-States   <=50

In [2]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 32560 entries, 0 to 32559
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32560 non-null  int64
 1   workclass       32560 non-null  str  
 2   fnlwgt          32560 non-null  int64
 3   education       32560 non-null  str  
 4   education-num   32560 non-null  int64
 5   marital-status  32560 non-null  str  
 6   occupation      32560 non-null  str  
 7   relationship    32560 non-null  str  
 8   race            32560 non-null  str  
 9   sex             32560 non-null  str  
 10  capital-gain    32560 non-null  int64
 11  capital-loss    32560 non-null  int64
 12  hours-per-week  32560 non-null  int64
 13  native-country  32560 non-null  str  
 14  income          32560 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB
None


In [3]:
print(df['income'].value_counts())

income
<=50K    24719
>50K      7841
Name: count, dtype: int64


In [4]:
for col in df.columns:
    print(f"Column: {col}")
    print(df[col].value_counts())
    print("\n") 

Column: age
age
36    898
31    888
34    886
23    877
35    876
     ... 
83      6
88      3
85      3
86      1
87      1
Name: count, Length: 73, dtype: int64


Column: workclass
workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1297
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64


Column: fnlwgt
fnlwgt
123011    13
164190    13
203488    13
148995    12
121124    12
          ..
129912     1
255835     1
34066      1
84661      1
257302     1
Name: count, Length: 21647, dtype: int64


Column: education
education
HS-grad         10501
Some-college     7291
Bachelors        5354
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333


In [5]:
df = df.apply(lambda x: x.str.strip() if x.dtype == 'str' else x)

In [6]:
import numpy as np

df = df.replace('?', np.nan)

In [7]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 32560 entries, 0 to 32559
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32560 non-null  int64
 1   workclass       30724 non-null  str  
 2   fnlwgt          32560 non-null  int64
 3   education       32560 non-null  str  
 4   education-num   32560 non-null  int64
 5   marital-status  32560 non-null  str  
 6   occupation      30717 non-null  str  
 7   relationship    32560 non-null  str  
 8   race            32560 non-null  str  
 9   sex             32560 non-null  str  
 10  capital-gain    32560 non-null  int64
 11  capital-loss    32560 non-null  int64
 12  hours-per-week  32560 non-null  int64
 13  native-country  31977 non-null  str  
 14  income          32560 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB
None


In [8]:
df = df.drop(columns = ['fnlwgt', 'education-num'])

In [9]:
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_cols = df.select_dtypes(include = 'str').columns.tolist()


In [11]:
categorical_cols.remove('income')

In [13]:
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

In [14]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split   

X = df.drop(columns = ['income'])
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [16]:

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop = 'first'))
])

In [17]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced')
}

fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)])
    
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    test_acc = accuracy_score(y_test, pipe.predict(X_test))
    print(f"{name} - Train Accuracy: {train_acc:.4f}, Test Accuracy: {test_acc:.4f}")

Logistic Regression - Train Accuracy: 0.8075, Test Accuracy: 0.8120
Random Forest - Train Accuracy: 0.9752, Test Accuracy: 0.8492
Decision Tree - Train Accuracy: 0.9722, Test Accuracy: 0.8213


In [22]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv = 5, scoring='f1')
    print(f"{name} - Cross-Validation F1-Score: {scores.mean():.4f} ± {scores.std():.4f}")

/home/bibek/.local/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Logistic Regression - Cross-Validation F1-Score: 0.6779 ± 0.0070


/home/bibek/.local/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Random Forest - Cross-Validation F1-Score: 0.6535 ± 0.0076


/home/bibek/.local/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Decision Tree - Cross-Validation F1-Score: 0.6203 ± 0.0127


In [23]:
from sklearn.model_selection import GridSearchCV

model_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

prarm_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(model_pipe, param_grid=prarm_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)
best_model = grid.best_estimator_
print(best_model.score(X_test, y_test))

/home/bibek/.local/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/bibek/.local/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/bibek/.local/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/bibek/.local/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/bibek/

{'model__C': 10}
0.678131448214852
0.8123464373464373
